In [ ]:
"""
Complete Hyperparameter Search for Colab Pro
=============================================

Runtime: ~6 hours on A100 for 15 configurations
Colab Pro features used:
- Background execution (can close browser)
- Long runtime (up to 24 hours)
- Save to Google Drive for persistence

Instructions:
1. Runtime → Change runtime type → A100 GPU
2. Run all cells
3. Can close browser - it will keep running
4. Results saved to Google Drive
"""

'\nComplete Hyperparameter Search for Colab Pro\n=============================================\n\nRuntime: ~6 hours on A100 for 15 configurations\nColab Pro features used:\n- Background execution (can close browser)\n- Long runtime (up to 24 hours)\n- Save to Google Drive for persistence\n\nInstructions:\n1. Runtime → Change runtime type → A100 GPU\n2. Run all cells\n3. Can close browser - it will keep running\n4. Results saved to Google Drive\n'

### Run First - Correct Numpy

In [ ]:
# Uninstall conflicting packages
!pip uninstall -y numpy pandas bitsandbytes datasets transformers accelerate peft trl

# Install compatible numpy FIRST
!pip install "numpy>=1.26.0,<2.0.0"

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: pandas 2.2.2
Uninstalling pandas-2.2.2:
  Successfully uninstalled pandas-2.2.2
Found existing installation: bitsandbytes 0.49.1
Uninstalling bitsandbytes-0.49.1:
  Successfully uninstalled bitsandbytes-0.49.1
Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0
Found existing installation: transformers 4.44.2
Uninstalling transformers-4.44.2:
  Successfully uninstalled transformers-4.44.2
Found existing installation: accelerate 0.33.0
Uninstalling accelerate-0.33.0:
  Successfully uninstalled accelerate-0.33.0
Found existing installation: peft 0.12.0
Uninstalling peft-0.12.0:
  Successfully uninstalled peft-0.12.0
Found existing installation: trl 0.9.6
Uninstalling trl-0.9.6:
  Successfully uninstalled trl-0.9.6
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x

In [ ]:
# Then install everything else
!pip install -q \
    "transformers==4.44.2" \
    "accelerate==0.33.0" \
    "peft==0.12.0" \
    "trl==0.9.6" \
    "bitsandbytes>=0.44.0" \
    "datasets>=2.19.0" \
    "pandas"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 148.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 50.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.0 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.0 which is incompatible.
dask-cudf-cu12 25.10.0 requires pandas<2.4.0dev0,>=2.0, but you have pandas 3.0.0 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.0 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you 

In [ ]:
import numpy as np
import pandas as pd
import bitsandbytes as bnb
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

print(f"numpy: {np.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

numpy: 1.26.4
GPU: NVIDIA A100-SXM4-40GB


### Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
SAVE_DIR = '/content/drive/MyDrive/lora_hp_search'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Results will save to: {SAVE_DIR}")

Results will save to: /content/drive/MyDrive/lora_hp_search


### Load in Data

In [ ]:
import json
def load_jsonl(filepath):
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

DATA_FILE = "/content/drive/MyDrive/lora_hp_search/cot_research_questions_total.jsonl"
raw_data = load_jsonl(DATA_FILE)
print(f"Loaded {len(raw_data)} samples")

# Preview
if raw_data:
    sample = raw_data[0]
    print(f"\nKeys: {list(sample.keys())}")

Loaded 2959 samples

Keys: ['paper_id', 'paper_title', 'paper_core_content', 'paper_token_estimate', 'similar_paper_id', 'similar_paper_title', 'similar_research_question', 'similarity_score', 'similar_core_content', 'similar_token_estimate', 'generated_research_question']


### HuggingFace Login

In [ ]:
from huggingface_hub import login
login()

### Imports

In [ ]:
import gc
import json
import torch
import pandas as pd
from datetime import datetime
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    BitsAndBytesConfig,
    EarlyStoppingCallback,
)
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

# Verify GPU
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU: NVIDIA A100-SXM4-40GB
Memory: 42.5 GB


### Configurations to Test

In [ ]:
CONFIGS = [
    # ID 0: Baseline (conservative)
    {"lora_r": 32, "lora_alpha": 64, "lora_dropout": 0.1, "include_mlp": True,
     "learning_rate": 5e-5, "weight_decay": 0.05, "warmup_ratio": 0.1},

    # ID 1: Higher capacity
    {"lora_r": 64, "lora_alpha": 128, "lora_dropout": 0.1, "include_mlp": True,
     "learning_rate": 5e-5, "weight_decay": 0.05, "warmup_ratio": 0.1},

    # ID 2: Lower learning rate
    {"lora_r": 64, "lora_alpha": 128, "lora_dropout": 0.1, "include_mlp": True,
     "learning_rate": 2e-5, "weight_decay": 0.05, "warmup_ratio": 0.1},

    # ID 3: Higher learning rate
    {"lora_r": 64, "lora_alpha": 128, "lora_dropout": 0.1, "include_mlp": True,
     "learning_rate": 1e-4, "weight_decay": 0.05, "warmup_ratio": 0.1},

    # ID 4: Less regularization
    {"lora_r": 64, "lora_alpha": 128, "lora_dropout": 0.05, "include_mlp": True,
     "learning_rate": 5e-5, "weight_decay": 0.01, "warmup_ratio": 0.1},

    # ID 5: More regularization
    {"lora_r": 64, "lora_alpha": 128, "lora_dropout": 0.15, "include_mlp": True,
     "learning_rate": 5e-5, "weight_decay": 0.1, "warmup_ratio": 0.1},

    # ID 6: Attention only (no MLP)
    {"lora_r": 64, "lora_alpha": 128, "lora_dropout": 0.1, "include_mlp": False,
     "learning_rate": 5e-5, "weight_decay": 0.05, "warmup_ratio": 0.1},

    # ID 7: Lower alpha ratio
    {"lora_r": 64, "lora_alpha": 64, "lora_dropout": 0.1, "include_mlp": True,
     "learning_rate": 5e-5, "weight_decay": 0.05, "warmup_ratio": 0.1},

    # ID 8: Very high capacity
    {"lora_r": 128, "lora_alpha": 256, "lora_dropout": 0.1, "include_mlp": True,
     "learning_rate": 2e-5, "weight_decay": 0.05, "warmup_ratio": 0.1},

    # ID 9: Small model, low LR
    {"lora_r": 32, "lora_alpha": 64, "lora_dropout": 0.1, "include_mlp": True,
     "learning_rate": 2e-5, "weight_decay": 0.05, "warmup_ratio": 0.1},

    # ID 10: Small + attention only
    {"lora_r": 32, "lora_alpha": 64, "lora_dropout": 0.1, "include_mlp": False,
     "learning_rate": 5e-5, "weight_decay": 0.05, "warmup_ratio": 0.1},

    # ID 11: High capacity + high LR
    {"lora_r": 128, "lora_alpha": 256, "lora_dropout": 0.1, "include_mlp": True,
     "learning_rate": 5e-5, "weight_decay": 0.05, "warmup_ratio": 0.1},

    # ID 12: Shorter warmup
    {"lora_r": 64, "lora_alpha": 128, "lora_dropout": 0.1, "include_mlp": True,
     "learning_rate": 5e-5, "weight_decay": 0.05, "warmup_ratio": 0.03},

    # ID 13: Longer warmup
    {"lora_r": 64, "lora_alpha": 128, "lora_dropout": 0.1, "include_mlp": True,
     "learning_rate": 5e-5, "weight_decay": 0.05, "warmup_ratio": 0.15},

    # ID 14: Balanced alternative
    {"lora_r": 48, "lora_alpha": 96, "lora_dropout": 0.1, "include_mlp": True,
     "learning_rate": 3e-5, "weight_decay": 0.03, "warmup_ratio": 0.1},
]

# Add fixed params and IDs
for i, config in enumerate(CONFIGS):
    config["config_id"] = i
    config["num_epochs"] = 2
    config["batch_size"] = 2
    config["gradient_accumulation_steps"] = 4
    config["max_seq_length"] = 2048

print(f"Total configurations to test: {len(CONFIGS)}")
print(f"Estimated time: {len(CONFIGS) * 25 / 60:.1f} hours on A100")

Total configurations to test: 15
Estimated time: 6.2 hours on A100


### Prepare Data

In [ ]:
SYSTEM_PROMPT = """You are an expert quantitative finance researcher. Given a research question, develop a comprehensive research plan with:
1. Problem formulation
2. Mathematical framework
3. Methodology
4. Data requirements
5. Expected contributions"""

def load_jsonl(filepath):
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

def format_sample(sample, max_content_len=3000):
    question = sample.get('generated_research_question', '')
    title = sample.get('paper_title', 'Research')
    content = sample.get('paper_core_content', '')

    if len(content) > max_content_len:
        content = content[:max_content_len] + "..."

    return {"text": f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|

{SYSTEM_PROMPT}<|eot_id|><|start_header_id|>user<|end_header_id|>

Develop a research approach for: {question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

# Research Framework: {title}

{content}<|eot_id|>"""}

# Load data
raw_data = load_jsonl(DATA_FILE)
processed = [format_sample(s) for s in raw_data]
dataset = Dataset.from_list(processed)

# Split
train_size = int(len(dataset) * 0.9)
train_dataset = dataset.select(range(train_size))
eval_dataset = dataset.select(range(train_size, len(dataset)))

print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")

Train: 2663 | Eval: 296


### Training Function

In [ ]:
### Define Base Model - Using 3.1 8B Instruct
BASE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"

In [ ]:
def train_config(config, train_dataset, eval_dataset):
    """Train a single configuration and return results."""

    gc.collect()
    torch.cuda.empty_cache()

    config_name = f"r{config['lora_r']}_lr{config['learning_rate']}_mlp{config['include_mlp']}"
    print(f"\n{'='*60}")
    print(f"Config {config['config_id']}: {config_name}")
    print(f"{'='*60}")

    results = {"config_id": config["config_id"], "config_name": config_name, **config}

    try:
        # Quantization
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )

        # Load model
        model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
        )
        model.config.use_cache = False

        tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = "right"

        # LoRA config
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"]
        if config["include_mlp"]:
            target_modules += ["gate_proj", "up_proj", "down_proj"]

        peft_config = LoraConfig(
            r=config["lora_r"],
            lora_alpha=config["lora_alpha"],
            lora_dropout=config["lora_dropout"],
            bias="none",
            task_type="CAUSAL_LM",
            target_modules=target_modules,
        )

        # Training args
        training_args = TrainingArguments(
            output_dir=f"./results/{config_name}",
            num_train_epochs=config["num_epochs"],
            per_device_train_batch_size=config["batch_size"],
            per_device_eval_batch_size=config["batch_size"],
            gradient_accumulation_steps=config["gradient_accumulation_steps"],
            optim="paged_adamw_32bit",
            learning_rate=config["learning_rate"],
            weight_decay=config["weight_decay"],
            warmup_ratio=config["warmup_ratio"],
            lr_scheduler_type="cosine",
            max_grad_norm=0.5,
            bf16=True,
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
            logging_steps=50,
            eval_strategy="steps",
            eval_steps=100,
            save_strategy="no",  # Don't save checkpoints to save space
            report_to="none",
            seed=42,
        )

        # Train
        trainer = SFTTrainer(
            model=model,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            peft_config=peft_config,
            dataset_text_field="text",
            max_seq_length=config["max_seq_length"],
            tokenizer=tokenizer,
            args=training_args,
            packing=False,
        )

        train_result = trainer.train()
        eval_result = trainer.evaluate()

        # Record results
        results["eval_loss"] = eval_result["eval_loss"]
        results["train_loss"] = train_result.training_loss
        results["runtime_min"] = train_result.metrics.get("train_runtime", 0) / 60
        results["status"] = "success"

        print(f"✓ eval_loss: {results['eval_loss']:.4f}")
        print(f"✓ train_loss: {results['train_loss']:.4f}")
        print(f"✓ runtime: {results['runtime_min']:.1f} min")

        # Cleanup
        del model, trainer
        gc.collect()
        torch.cuda.empty_cache()

    except Exception as e:
        results["eval_loss"] = float("inf")
        results["status"] = "failed"
        results["error"] = str(e)
        print(f"✗ Failed: {e}")

        gc.collect()
        torch.cuda.empty_cache()

    return results

### Run Full Search

In [ ]:
print("="*60)
print("STARTING HYPERPARAMETER SEARCH")
print(f"Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*60)

all_results = []
results_file = os.path.join(SAVE_DIR, "hp_search_results.csv")

for i, config in enumerate(CONFIGS):
    print(f"\n[{i+1}/{len(CONFIGS)}]", end="")

    result = train_config(config, train_dataset, eval_dataset)
    all_results.append(result)

    # Save after EVERY config (in case of crash)
    df = pd.DataFrame(all_results)
    df.to_csv(results_file, index=False)
    print(f"Progress saved to {results_file}")

print("\n" + "="*60)
print("SEARCH COMPLETE")
print(f"Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*60)

### Results

In [ ]:
import os
results_file = os.path.join(SAVE_DIR, "hp_search_results.csv")

In [ ]:
import json
df = pd.read_csv(results_file)
df_success = df[df["status"] == "success"].sort_values("eval_loss")

print("\n" + "="*60)
print("RESULTS RANKED BY EVAL LOSS")
print("="*60)

for i, (_, row) in enumerate(df_success.iterrows()):
    marker = "" if i == 0 else "  "
    print(f"{marker} {i+1}. eval_loss={row['eval_loss']:.4f} | "
          f"r={row['lora_r']}, lr={row['learning_rate']:.0e}, "
          f"mlp={row['include_mlp']}, wd={row['weight_decay']}")

# Best configuration
best = df_success.iloc[0]

print("\n" + "="*60)
print("BEST CONFIGURATION")
print("="*60)
print(f"""
lora_r: {best['lora_r']}
lora_alpha: {best['lora_alpha']}
lora_dropout: {best['lora_dropout']}
include_mlp: {best['include_mlp']}
learning_rate: {best['learning_rate']}
weight_decay: {best['weight_decay']}
warmup_ratio: {best['warmup_ratio']}

eval_loss: {best['eval_loss']:.4f}
train_loss: {best['train_loss']:.4f}
""")

# Save best config
best_config = best.to_dict()
with open(os.path.join(SAVE_DIR, "best_config.json"), "w") as f:
    json.dump(best_config, f, indent=2, default=str)

print(f"Best config saved to: {SAVE_DIR}/best_config.json")


RESULTS RANKED BY EVAL LOSS
 1. eval_loss=1.5856 | r=64, lr=5e-05, mlp=True, wd=0.1
   2. eval_loss=1.5856 | r=64, lr=5e-05, mlp=True, wd=0.05
   3. eval_loss=1.5858 | r=64, lr=5e-05, mlp=True, wd=0.01
   4. eval_loss=1.5859 | r=64, lr=5e-05, mlp=True, wd=0.05
   5. eval_loss=1.5860 | r=64, lr=5e-05, mlp=True, wd=0.05
   6. eval_loss=1.5869 | r=128, lr=2e-05, mlp=True, wd=0.05
   7. eval_loss=1.5879 | r=128, lr=5e-05, mlp=True, wd=0.05
   8. eval_loss=1.5880 | r=32, lr=5e-05, mlp=True, wd=0.05
   9. eval_loss=1.5883 | r=64, lr=5e-05, mlp=True, wd=0.05
   10. eval_loss=1.5900 | r=48, lr=3e-05, mlp=True, wd=0.03
   11. eval_loss=1.5900 | r=64, lr=1e-04, mlp=True, wd=0.05
   12. eval_loss=1.5921 | r=64, lr=2e-05, mlp=True, wd=0.05
   13. eval_loss=1.5999 | r=32, lr=2e-05, mlp=True, wd=0.05
   14. eval_loss=1.6012 | r=64, lr=5e-05, mlp=False, wd=0.05
   15. eval_loss=1.6084 | r=32, lr=5e-05, mlp=False, wd=0.05

BEST CONFIGURATION

lora_r: 64
lora_alpha: 128
lora_dropout: 0.15
include_mlp:

In [ ]:
def train_config_final(config, train_dataset, eval_dataset, save_dir="/content/final_model"):
    """Train a single configuration and return results."""

    gc.collect()
    torch.cuda.empty_cache()

    config_name = f"r{config['lora_r']}_lr{config['learning_rate']}_mlp{config['include_mlp']}"
    print(f"\n{'='*60}")
    print(f"Config {config['config_id']}: {config_name}")
    print(f"{'='*60}")

    results = {"config_id": config["config_id"], "config_name": config_name, **config}

    try:
        # Quantization
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )

        # Load model
        model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
        )
        model.config.use_cache = False

        tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = "right"

        # LoRA config
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"]
        if config["include_mlp"]:
            target_modules += ["gate_proj", "up_proj", "down_proj"]

        peft_config = LoraConfig(
            r=config["lora_r"],
            lora_alpha=config["lora_alpha"],
            lora_dropout=config["lora_dropout"],
            bias="none",
            task_type="CAUSAL_LM",
            target_modules=target_modules,
        )

        # Training args
        training_args = TrainingArguments(
            output_dir=f"./results/{config_name}",
            num_train_epochs=config["num_epochs"],
            per_device_train_batch_size=config["batch_size"],
            per_device_eval_batch_size=config["batch_size"],
            gradient_accumulation_steps=config["gradient_accumulation_steps"],
            optim="paged_adamw_32bit",
            learning_rate=config["learning_rate"],
            weight_decay=config["weight_decay"],
            warmup_ratio=config["warmup_ratio"],
            lr_scheduler_type="cosine",
            max_grad_norm=0.5,
            bf16=True,
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
            logging_steps=50,
            eval_strategy="steps",
            eval_steps=100,
            save_strategy="no",
            report_to="none",
            seed=42,
        )

        # Train
        trainer = SFTTrainer(
            model=model,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            peft_config=peft_config,
            dataset_text_field="text",
            max_seq_length=config["max_seq_length"],
            tokenizer=tokenizer,
            args=training_args,
            packing=False,
        )

        train_result = trainer.train()
        eval_result = trainer.evaluate()

        # Record results
        results["eval_loss"] = eval_result["eval_loss"]
        results["train_loss"] = train_result.training_loss
        results["runtime_min"] = train_result.metrics.get("train_runtime", 0) / 60
        results["status"] = "success"

        print(f"✓ eval_loss: {results['eval_loss']:.4f}")
        print(f"✓ train_loss: {results['train_loss']:.4f}")
        print(f"✓ runtime: {results['runtime_min']:.1f} min")

        # ============================================================
        # SAVE MODEL AND TOKENIZER
        # ============================================================
        import os

        # Create save directories
        lora_save_path = os.path.join(save_dir, "lora_adapters")
        merged_save_path = os.path.join(save_dir, "merged_model")
        os.makedirs(lora_save_path, exist_ok=True)
        os.makedirs(merged_save_path, exist_ok=True)

        print(f"\n{'='*60}")
        print("SAVING MODEL")
        print(f"{'='*60}")

        # Save LoRA adapters
        trainer.model.save_pretrained(lora_save_path)
        tokenizer.save_pretrained(lora_save_path)
        print(f"✓ LoRA adapters saved to: {lora_save_path}")

        # Save merged model (LoRA weights merged into base model)
        print("Merging LoRA weights with base model...")
        merged_model = trainer.model.merge_and_unload()
        merged_model.save_pretrained(merged_save_path)
        tokenizer.save_pretrained(merged_save_path)
        print(f"✓ Merged model saved to: {merged_save_path}")

        # Save training config for reference
        import json
        config_save_path = os.path.join(save_dir, "training_config.json")
        with open(config_save_path, "w") as f:
            json.dump(config, f, indent=2)
        print(f"✓ Training config saved to: {config_save_path}")

        results["lora_path"] = lora_save_path
        results["merged_path"] = merged_save_path
        # ============================================================

        # Cleanup
        del merged_model
        gc.collect()
        torch.cuda.empty_cache()

    except Exception as e:
        results["eval_loss"] = float("inf")
        results["status"] = "failed"
        results["error"] = str(e)
        print(f"✗ Failed: {e}")

        gc.collect()
        torch.cuda.empty_cache()

    return results

### Final Training With Best Config

In [ ]:
print("\n" + "="*60)
print("TRAINING FINAL MODEL WITH BEST CONFIGURATION")
print("="*60)

# Use best config but train for slightly longer
final_config = {
    "lora_r": int(best['lora_r']),
    "lora_alpha": int(best['lora_alpha']),
    "lora_dropout": float(best['lora_dropout']),
    "include_mlp": bool(best['include_mlp']),
    "learning_rate": float(best['learning_rate']),
    "weight_decay": float(best['weight_decay']),
    "warmup_ratio": float(best['warmup_ratio']),
    "num_epochs": 3,  # Train a bit longer for final model
    "batch_size": 2,
    "gradient_accumulation_steps": 4,
    "max_seq_length": 2048,
    "config_id": 999,
}

# Train final model
final_result = train_config_final(
    final_config,
    train_dataset,
    eval_dataset,
    save_dir="/content/drive/MyDrive/lora_hp_search"
)



TRAINING FINAL MODEL WITH BEST CONFIGURATION

Config 999: r64_lr5e-05_mlpTrue


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

... (10 outputs left out)

Step,Training Loss,Validation Loss
100,1.650800,1.629693
200,1.626500,1.605818
300,1.574000,1.591472
400,1.496700,1.597804
500,1.486800,1.588791
600,1.464400,1.586905
700,1.370200,1.606874
800,1.335600,1.608591
900,1.319400,1.608180


✓ eval_loss: 1.6071
✓ train_loss: 1.4965
✓ runtime: 73.0 min

SAVING MODEL
✓ LoRA adapters saved to: /content/drive/MyDrive/lora_hp_search/lora_adapters
Merging LoRA weights with base model...


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:336: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


✓ Merged model saved to: /content/drive/MyDrive/lora_hp_search/merged_model
✓ Training config saved to: /content/drive/MyDrive/lora_hp_search/training_config.json
